# ThyroidXL Stage-2 Oracle ROI Experiment

This notebook compares the existing automatic Stage-1-derived Stage-2 crop against an oracle crop derived from the ThyroidXL ground-truth mask. The GT is used only to select the crop; Stage 2 still receives only the ultrasound image.


In [1]:
import sys
from pathlib import Path
import numpy as np 

MODULE_PARENT = Path("/content/sgh-segmodel/code")

if str(MODULE_PARENT) not in sys.path:
    sys.path.insert(0, str(MODULE_PARENT))

import segmentation_models_pytorch_4TorchLessThan120 as smp

/Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/step2to4_train_validate_inference/segmentation_models_pytorch_4TorchLessThan120/__init__.py:14: UserWarning: segmentation_models_pytorch_4TorchLessThan120 does not suppose timm_efficientnet_encoders
  warnings.warn('segmentation_models_pytorch_4TorchLessThan120 does not suppose timm_efficientnet_encoders')


In [5]:
C1_SIZE = 256 

def thyroidxl_preprocess(
    image_path: Path,
    outputsize: C1_SIZE,
    remove_black_edges: bool = True,
):
    """
    ThyroidXL-safe replacement for TNSCUI_preprocess. 
    (original function in utils:
        removes irrelevant edges based on a threshold system of averaged pixels row-wise; the default output_size is 256 x 256
        returns the cut image tensor, coordinates of the min and max x and y FROM ORIGINAL IMAGE respectively that the extracted image without irrelevant
        areas contains  

    Returns
    -------
    processed_tensor:
        Float tensor with shape [output_size, output_size].

    cut_shape:
        Shape of the retained image before resizing.

    original_shape:
        Original image shape: (height, width).

    location:
        Coordinates of the retained image in the original image:
        [row_start, row_end, col_start, col_end].
    """

    # Force the image into one grayscale channel.
    with Image.open(image_path) as image:
        image = image.convert("L")
        image_array = np.asarray(image, dtype=np.float32)

    original_shape = image_array.shape

    if image_array.ndim != 2:
        raise ValueError(
            f"Expected a 2-D grayscale image, got {image_array.shape} "
            f"for {image_path.name}"
        )

    if remove_black_edges:
        # Detect rows and columns that contain meaningful ultrasound content.
        #
        # A small threshold is used instead of requiring pixels to be exactly
        # zero because ultrasound borders may contain compression noise.
        foreground_threshold = 5.0

        valid_rows = np.where(
            np.mean(image_array, axis=1) > foreground_threshold
        )[0]

        valid_cols = np.where(
            np.mean(image_array, axis=0) > foreground_threshold
        )[0]

        if len(valid_rows) > 0 and len(valid_cols) > 0:
            row_start = int(valid_rows[0])
            row_end = int(valid_rows[-1]) + 1

            col_start = int(valid_cols[0])
            col_end = int(valid_cols[-1]) + 1
        else:
            # Fall back to the complete image if no foreground is found.
            row_start = 0
            row_end = original_shape[0]
            col_start = 0
            col_end = original_shape[1]

    else:
        row_start = 0
        row_end = original_shape[0]
        col_start = 0
        col_end = original_shape[1]

    cropped_image = image_array[
        row_start:row_end,
        col_start:col_end,
    ]

    if cropped_image.size == 0:
        raise ValueError(
            f"Black-edge removal produced an empty image for "
            f"{image_path.name}"
        )

    cut_shape = cropped_image.shape
    location = [row_start, row_end, col_start, col_end]

    # Resize to the stage-1 
    processed_array = resize(
        cropped_image,
        (outputsize, outputsize),
        order=3,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    # Match the common neural-network image range.
    if processed_array.max() > 1.0:
        processed_array /= 255.0

    processed_tensor = torch.from_numpy(processed_array)

    return processed_tensor, cut_shape, original_shape, location

In [6]:

import csv
import os
import traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import ttach as tta
from PIL import Image
from skimage.measure import label as sklabel
from skimage.measure import regionprops
from skimage.transform import resize
import segmentation_models_pytorch_4TorchLessThan120 as smp


# LOAD DATA CONFIGS 

PROJECT_ROOT = Path(
    "/Users/JanayeCheong/Documents/radiomics_segmentation_models/"
    "TNSCUI2020-Seg-Rank1st"
)

IMG_DIR = PROJECT_ROOT / "train_thyroidXL" / "raw_images"
MASK_DIR = PROJECT_ROOT / "train_thyroidXL" / "masks"

WEIGHT_C1 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage1_trained_on_size_256.pkl"
)

WEIGHT_C2 = (
    PROJECT_ROOT
    / "weigh_and_id"
    / "TNSCUI"
    / "fold1_stage2_trained_on_size_512.pkl"
)

OUTPUT_DIR = PROJECT_ROOT / "oracle_roi_experiment"
NORMAL_DIR = OUTPUT_DIR / "normal_stage1_roi_masks"
ORACLE_DIR = OUTPUT_DIR / "oracle_gt_roi_masks"
OVERLAY_DIR = OUTPUT_DIR / "comparison_overlays"
METRICS_CSV = OUTPUT_DIR / "oracle_roi_metrics.csv"

C1_SIZE = 256
C2_SIZE = 512

C1_TTA = True
C2_TTA = True
USE_C2 = True

ORIMG = False
C1_THRESHOLD = 0.5
C2_THRESHOLD = 0.5
C2_RESIZE_ORDER = 0

SAVE_OVERLAYS = True
CONTINUE_ON_ERROR = True

# Start with known irregular cases. Set to None to run the full dataset.
SELECTED_IMAGE_STEMS = {
    "00000155_A2330FD3_0",
    "00000139_217557BB_1",
}

SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}



# Metrics 

def get_iou(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate intersection over union for two binary arrays --> corresponds to ."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    union = np.logical_or(prediction, ground_truth).sum()

    if union == 0:
        return 1.0

    return float(intersection / union)


def get_dsc(prediction: np.ndarray, ground_truth: np.ndarray) -> float:
    """Calculate Dice similarity coefficient for two binary arrays."""
    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    intersection = np.logical_and(prediction, ground_truth).sum()
    denominator = prediction.sum() + ground_truth.sum()

    if denominator == 0:
        return 1.0

    return float((2.0 * intersection) / denominator)


def largest_connected_component(binary_mask: np.ndarray) -> np.ndarray:
    """Retain only the largest foreground connected component."""
    binary_mask = binary_mask.astype(bool)

    if binary_mask.sum() == 0:
        return binary_mask.astype(np.float32)

    labeled_img, num_components = sklabel(
        binary_mask,
        connectivity=1,
        background=0,
        return_num=True,
    )

    if num_components == 1:
        return binary_mask.astype(np.float32)

    component_sizes = [
        np.sum(labeled_img == component_label)
        for component_label in range(1, num_components + 1)
    ]

    largest_label = int(np.argmax(component_sizes)) + 1
    return (labeled_img == largest_label).astype(np.float32)


def calculate_stage2_roi(
    stage1_mask: np.ndarray,
    c1_size: int = 256,
) -> Tuple[int, int, int, int]:
    """
    Calculate the expanded square ROI used as input to Stage 2.

    Returns
    -------
    row_min, row_max, col_min, col_max
    """
    if stage1_mask.sum() == 0:
        min_row, min_col, max_row, max_col = 0, 0, c1_size, c1_size
    else:
        region = regionprops(stage1_mask.astype(np.uint8))[0]
        min_row, min_col, max_row, max_col = region.bbox

    row_center = (max_row + min_row) // 2
    col_center = (max_col + min_col) // 2
    max_length = max(max_row - min_row, max_col - min_col)

    large_roi_threshold = int((c1_size / 256) * 80)
    large_roi_margin = int((c1_size / 256) * 19)
    small_roi_margin = int((c1_size / 256) * 31)

    if max_length > large_roi_threshold:
        expansion = large_roi_margin + max_length // 2
    else:
        expansion = small_roi_margin + max_length // 2

    row_min = max(0, row_center - expansion)
    row_max = min(c1_size, row_center + expansion)
    col_min = max(0, col_center - expansion)
    col_max = min(c1_size, col_center + expansion)

    # IN case the crop is empty    
    if row_max <= row_min or col_max <= col_min:
        return 0, c1_size, 0, c1_size

    return row_min, row_max, col_min, col_max


# save files in order 

def discover_images(image_dir: Path) -> List[Path]:
    """Return every supported image file recursively, in stable order."""
    files = [
        path
        for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return sorted(files, key=lambda path: str(path).lower())


def build_mask_index(mask_dir: Path) -> Dict[str, Path]:
    """
    Index masks by filename stem, case-insensitively.

    The image and mask may have different filename extensions, but their stems
    must match
    """
    index: Dict[str, Path] = {}

    for path in discover_images(mask_dir):
        key = path.stem.lower()

        if key in index:
            raise ValueError(
                f"Duplicate mask stem '{path.stem}' found:\n"
                f"  {index[key]}\n"
                f"  {path}"
            )

        index[key] = path

    return index


def load_binary_mask(mask_path: Path, expected_shape: Tuple[int, int]) -> np.ndarray:
    """Read a mask as grayscale and convert it to a binary NumPy array."""
    mask = Image.open(mask_path).convert("L")
    mask_array = np.asarray(mask, dtype=np.float32)

    if mask_array.shape != expected_shape:
        raise ValueError(
            f"Ground-truth mask shape {mask_array.shape} does not match "
            f"original image shape {expected_shape} for {mask_path.name}."
        )

    return (mask_array > 0).astype(np.float32)


def save_binary_mask(mask: np.ndarray, output_path: Path) -> None:
    """Save a binary mask as an 8-bit PNG."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    mask_uint8 = (mask.astype(bool).astype(np.uint8) * 255)
    Image.fromarray(mask_uint8, mode="L").save(output_path)


def save_overlay(
    image_path: Path,
    prediction: np.ndarray,
    ground_truth: Optional[np.ndarray],
    output_path: Path,
) -> None:
    """
    Save an RGB overlay:
    - red: the prediction
    - green: the ground truth (from given mask)
    - yellow: overlap
    """
    original = Image.open(image_path).convert("L")
    base = np.asarray(original, dtype=np.float32)

    if base.max() > base.min():
        base = (base - base.min()) / (base.max() - base.min())
    else:
        base = np.zeros_like(base)

    rgb = np.stack([base, base, base], axis=-1)
    prediction_bool = prediction.astype(bool)

    rgb[prediction_bool, 0] = 1.0
    rgb[prediction_bool, 1] *= 0.35
    rgb[prediction_bool, 2] *= 0.35

    if ground_truth is not None:
        ground_truth_bool = ground_truth.astype(bool)
        rgb[ground_truth_bool, 1] = 1.0
        rgb[ground_truth_bool, 0] *= 0.35
        rgb[ground_truth_bool, 2] *= 0.35

        overlap = np.logical_and(prediction_bool, ground_truth_bool)
        rgb[overlap, 0] = 1.0
        rgb[overlap, 1] = 1.0
        rgb[overlap, 2] = 0.0

    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8), mode="RGB").save(
        output_path
    )


# ============================================================================
# Model loading and inference
# ============================================================================

def choose_device() -> torch.device:
    """Prefer Apple MPS, then CUDA, then CPU."""
    if torch.backends.mps.is_available():
        return torch.device("mps")

    if torch.cuda.is_available():
        return torch.device("cuda")

    return torch.device("cpu")


def extract_state_dict(checkpoint):
    """
    Support either a direct state_dict or a checkpoint dictionary containing
    a state_dict/model_state_dict field.
    """
    if not isinstance(checkpoint, dict):
        return checkpoint

    if "state_dict" in checkpoint:
        return checkpoint["state_dict"]

    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]

    return checkpoint


def load_model(
    weight_path: Path,
    device: torch.device,
    transforms,
    use_tta: bool,
) -> torch.nn.Module:
    """Create a DeepLabV3+ model and load pretrained weights."""
    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")

    model = smp.DeepLabV3Plus(
        encoder_name="efficientnet-b6",
        encoder_weights=None,
        in_channels=1,
        classes=1,
    )

    # CPU deserialization is typically the safest for old checkpoints.
    checkpoint = torch.load(weight_path, map_location="cpu")
    state_dict = extract_state_dict(checkpoint)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        print(
            f"Strict loading failed for {weight_path.name}; "
            "retrying with strict=False."
        )
        print(exc)
        incompatible = model.load_state_dict(state_dict, strict=False)
        print("Missing keys:", incompatible.missing_keys)
        print("Unexpected keys:", incompatible.unexpected_keys)

    model = model.to(device)
    model.eval()

    if use_tta:
        model = tta.SegmentationTTAWrapper(
            model,
            transforms,
            merge_mode="mean",
        )
        model.eval()

    return model



def mask_original_to_stage1_space(
    ground_truth: np.ndarray,
    cut_shape: Tuple[int, int],
    location: List[int],
    output_size: int = C1_SIZE,
) -> np.ndarray:
    """
    Map the original-resolution GT mask into the same 256x256 coordinate
    system used by Stage 1.

    This is used ONLY to define the oracle Stage-2 crop.
    The GT mask itself is never given to Stage 2 as an input channel.
    """
    row_start, row_end, col_start, col_end = [int(v) for v in location]

    gt_cut = ground_truth[
        row_start:row_end,
        col_start:col_end,
    ]

    if gt_cut.size == 0:
        raise RuntimeError("GT crop is empty after preprocessing coordinates.")

    gt_256 = resize(
        gt_cut,
        (output_size, output_size),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )

    return (gt_256 > 0.5).astype(np.float32)


def restore_256_mask_to_original(
    mask_256: np.ndarray,
    cut_shape: Tuple[int, int],
    original_shape: Tuple[int, int],
    location: List[int],
) -> np.ndarray:
    """Restore a 256x256 binary mask to original ThyroidXL coordinates."""
    restored_cut_mask = resize(
        mask_256,
        tuple(int(v) for v in cut_shape),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    )

    restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    final_mask = np.zeros(
        tuple(int(v) for v in original_shape),
        dtype=np.float32,
    )

    row_start, row_end, col_start, col_end = [int(v) for v in location]

    target_shape = (
        row_end - row_start,
        col_end - col_start,
    )

    if restored_cut_mask.shape != target_shape:
        restored_cut_mask = resize(
            restored_cut_mask,
            target_shape,
            order=0,
            preserve_range=True,
            anti_aliasing=False,
        )
        restored_cut_mask = (restored_cut_mask > 0.5).astype(np.float32)

    final_mask[
        row_start:row_end,
        col_start:col_end,
    ] = restored_cut_mask

    return (final_mask > 0.5).astype(np.float32)


def run_stage2_on_roi(
    image_array_256: np.ndarray,
    roi_coords: Tuple[int, int, int, int],
    model_cascade2: torch.nn.Module,
    device: torch.device,
) -> np.ndarray:
    """
    Run the existing Stage-2 model on one ROI in Stage-1 (256x256) space.

    Returns a binary 256x256 mask containing only the Stage-2 prediction.
    """
    row_min, row_max, col_min, col_max = roi_coords

    roi = image_array_256[
        row_min:row_max,
        col_min:col_max,
    ]

    if roi.size == 0:
        raise RuntimeError(
            f"Stage-2 ROI is empty: {roi_coords}"
        )

    roi_original_shape = roi.shape

    roi_512 = resize(
        roi,
        (C2_SIZE, C2_SIZE),
        order=3,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    roi_tensor = (
        torch.from_numpy(roi_512)
        .unsqueeze(0)
        .unsqueeze(0)
        .to(device=device, dtype=torch.float32)
    )

    with torch.inference_mode():
        stage2_logits = model_cascade2(roi_tensor)
        stage2_probability = torch.sigmoid(stage2_logits)

    stage2_mask_512 = (
        stage2_probability
        .squeeze()
        .detach()
        .cpu()
        .numpy()
        > C2_THRESHOLD
    ).astype(np.float32)

    stage2_mask_roi = resize(
        stage2_mask_512,
        roi_original_shape,
        order=C2_RESIZE_ORDER,
        preserve_range=True,
        anti_aliasing=False,
    )

    stage2_mask_roi = (
        stage2_mask_roi > 0.5
    ).astype(np.float32)

    full_256 = np.zeros(
        (C1_SIZE, C1_SIZE),
        dtype=np.float32,
    )

    full_256[
        row_min:row_max,
        col_min:col_max,
    ] = stage2_mask_roi

    return full_256


def run_oracle_roi_experiment(
    image_path: Path,
    ground_truth: np.ndarray,
    model_cascade1: torch.nn.Module,
    model_cascade2: torch.nn.Module,
    device: torch.device,
):
    """
    Compare two Stage-2 conditions on the same ThyroidXL image:

    1. NORMAL:
       Stage-1 prediction -> Stage-1 bbox/margin -> Stage 2.

    2. ORACLE:
       Ground-truth mask -> GT bbox/margin -> Stage 2.

    The GT is used only to choose the oracle crop.
    Stage 2 still receives only the ultrasound image.
    """
    (
        processed_img,
        cut_shape,
        original_shape,
        location,
    ) = thyroidxl_preprocess(
        image_path,
        outputsize=C1_SIZE,
        remove_black_edges=not ORIMG,
    )

    image_tensor = (
        processed_img
        .unsqueeze(0)
        .unsqueeze(0)
        .to(device=device, dtype=torch.float32)
    )

    image_array_256 = (
        processed_img
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    # ------------------------------------------------------------------
    # Stage 1: used only for the NORMAL cascade condition
    # ------------------------------------------------------------------
    with torch.inference_mode():
        stage1_logits = model_cascade1(image_tensor)
        stage1_probability = torch.sigmoid(stage1_logits)

    stage1_mask_256 = (
        stage1_probability
        .squeeze()
        .detach()
        .cpu()
        .numpy()
        > C1_THRESHOLD
    ).astype(np.float32)

    stage1_mask_256 = largest_connected_component(
        stage1_mask_256
    )

    normal_roi = calculate_stage2_roi(
        stage1_mask_256,
        C1_SIZE,
    )

    normal_stage2_256 = run_stage2_on_roi(
        image_array_256,
        normal_roi,
        model_cascade2,
        device,
    )

    # ------------------------------------------------------------------
    # Oracle ROI: derive localization from GT, not Stage 1
    # ------------------------------------------------------------------
    gt_256 = mask_original_to_stage1_space(
        ground_truth,
        cut_shape,
        location,
        C1_SIZE,
    )

    gt_256 = largest_connected_component(
        gt_256
    )

    oracle_roi = calculate_stage2_roi(
        gt_256,
        C1_SIZE,
    )

    oracle_stage2_256 = run_stage2_on_roi(
        image_array_256,
        oracle_roi,
        model_cascade2,
        device,
    )

    # Restore both Stage-2 outputs to original ThyroidXL coordinates.
    normal_final = restore_256_mask_to_original(
        normal_stage2_256,
        cut_shape,
        original_shape,
        location,
    )

    oracle_final = restore_256_mask_to_original(
        oracle_stage2_256,
        cut_shape,
        original_shape,
        location,
    )

    stage1_original = restore_256_mask_to_original(
        stage1_mask_256,
        cut_shape,
        original_shape,
        location,
    )

    return {
        "stage1_original": stage1_original,
        "normal_final": normal_final,
        "oracle_final": oracle_final,
        "normal_roi_256": normal_roi,
        "oracle_roi_256": oracle_roi,
    }


def save_oracle_comparison_overlay(
    image_path: Path,
    ground_truth: np.ndarray,
    normal_prediction: np.ndarray,
    oracle_prediction: np.ndarray,
    output_path: Path,
) -> None:
    """
    Save a simple 3-panel comparison:
        1. GT
        2. normal Stage-1-derived ROI -> Stage 2
        3. GT-oracle ROI -> Stage 2

    Purple contour = GT.
    Filled overlay = prediction.
    """
    import matplotlib.pyplot as plt

    with Image.open(image_path) as img:
        image = np.asarray(img.convert("L"), dtype=np.float32)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    panels = [
        ("Ground truth", ground_truth),
        ("Normal Stage-1 ROI -> Stage 2", normal_prediction),
        ("Oracle GT ROI -> Stage 2", oracle_prediction),
    ]

    for ax, (title, mask) in zip(axes, panels):
        ax.imshow(image, cmap="gray")

        if title != "Ground truth":
            ax.imshow(
                np.ma.masked_where(mask == 0, mask),
                alpha=0.40,
                vmin=0,
                vmax=1,
            )

        ax.contour(
            ground_truth,
            levels=[0.5],
            linewidths=1.5,
        )

        ax.set_title(title)
        ax.axis("off")

    fig.suptitle(image_path.name)
    fig.tight_layout()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig.savefig(
        output_path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.close(fig)


# ============================================================================
# Main oracle-ROI experiment
# ============================================================================

def main() -> None:

    for required_path in (
        IMG_DIR,
        MASK_DIR,
        WEIGHT_C1,
        WEIGHT_C2,
    ):
        if not required_path.exists():
            raise FileNotFoundError(
                f"Path not found: {required_path}"
            )

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    NORMAL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    ORACLE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    OVERLAY_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    device = choose_device()

    image_files = discover_images(
        IMG_DIR
    )

    if SELECTED_IMAGE_STEMS is not None:
        image_files = [
            p for p in image_files
            if p.stem in SELECTED_IMAGE_STEMS
        ]

    mask_index = build_mask_index(
        MASK_DIR
    )

    if not image_files:
        raise RuntimeError(
            "No images matched the selected cases."
        )

    tta_transforms = tta.Compose(
        [
            tta.VerticalFlip(),
            tta.HorizontalFlip(),
            tta.Rotate90(angles=[0, 180]),
        ]
    )

    print(
        f"Loading Stage-1 and Stage-2 models on {device}..."
    )

    model_cascade1 = load_model(
        WEIGHT_C1,
        device,
        tta_transforms,
        C1_TTA,
    )

    model_cascade2 = load_model(
        WEIGHT_C2,
        device,
        tta_transforms,
        C2_TTA,
    )

    results: List[Dict] = []

    for index, image_path in enumerate(
        image_files,
        start=1,
    ):

        mask_path = mask_index[
            image_path.stem.lower()
        ]

        try:
            with Image.open(image_path) as original_image:
                original_shape = (
                    original_image.height,
                    original_image.width,
                )

            ground_truth = load_binary_mask(
                mask_path,
                original_shape,
            )

            outputs = run_oracle_roi_experiment(
                image_path,
                ground_truth,
                model_cascade1,
                model_cascade2,
                device,
            )

            normal_final = outputs[
                "normal_final"
            ]

            oracle_final = outputs[
                "oracle_final"
            ]

            normal_iou = get_iou(
                normal_final,
                ground_truth,
            )

            normal_dsc = get_dsc(
                normal_final,
                ground_truth,
            )

            oracle_iou = get_iou(
                oracle_final,
                ground_truth,
            )

            oracle_dsc = get_dsc(
                oracle_final,
                ground_truth,
            )

            normal_path = (
                NORMAL_DIR
                / f"{image_path.stem}_normal.png"
            )

            oracle_path = (
                ORACLE_DIR
                / f"{image_path.stem}_oracle.png"
            )

            overlay_path = (
                OVERLAY_DIR
                / f"{image_path.stem}_comparison.png"
            )

            save_binary_mask(
                normal_final,
                normal_path,
            )

            save_binary_mask(
                oracle_final,
                oracle_path,
            )

            save_oracle_comparison_overlay(
                image_path,
                ground_truth,
                normal_final,
                oracle_final,
                overlay_path,
            )

            normal_roi = outputs[
                "normal_roi_256"
            ]

            oracle_roi = outputs[
                "oracle_roi_256"
            ]

            results.append(
                {
                    "image_name": image_path.name,
                    "normal_iou": normal_iou,
                    "normal_dsc": normal_dsc,
                    "oracle_iou": oracle_iou,
                    "oracle_dsc": oracle_dsc,
                    "delta_iou": oracle_iou - normal_iou,
                    "delta_dsc": oracle_dsc - normal_dsc,
                    "normal_roi_256": str(normal_roi),
                    "oracle_roi_256": str(oracle_roi),
                    "normal_prediction_path": str(normal_path),
                    "oracle_prediction_path": str(oracle_path),
                    "overlay_path": str(overlay_path),
                    "status": "ok",
                    "error": "",
                }
            )

            print(
                f"[{index}/{len(image_files)}] {image_path.name} | "
                f"normal DSC={normal_dsc:.3f} | "
                f"oracle DSC={oracle_dsc:.3f} | "
                f"delta={oracle_dsc - normal_dsc:+.3f}"
            )

        except Exception as exc:

            print(
                f"[{index}/{len(image_files)}] "
                f"FAILED {image_path.name}: {exc}"
            )

            traceback.print_exc()

            results.append(
                {
                    "image_name": image_path.name,
                    "normal_iou": "",
                    "normal_dsc": "",
                    "oracle_iou": "",
                    "oracle_dsc": "",
                    "delta_iou": "",
                    "delta_dsc": "",
                    "normal_roi_256": "",
                    "oracle_roi_256": "",
                    "normal_prediction_path": "",
                    "oracle_prediction_path": "",
                    "overlay_path": "",
                    "status": "failed",
                    "error": str(exc),
                }
            )

            if not CONTINUE_ON_ERROR:
                break

    fieldnames = [
        "image_name",
        "normal_iou",
        "normal_dsc",
        "oracle_iou",
        "oracle_dsc",
        "delta_iou",
        "delta_dsc",
        "normal_roi_256",
        "oracle_roi_256",
        "normal_prediction_path",
        "oracle_prediction_path",
        "overlay_path",
        "status",
        "error",
    ]

    with METRICS_CSV.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as csv_file:

        writer = csv.DictWriter(
            csv_file,
            fieldnames=fieldnames,
        )

        writer.writeheader()
        writer.writerows(results)

    print(
        f"\nDone. Comparison overlays: {OVERLAY_DIR}"
    )

    print(
        f"Metrics: {METRICS_CSV}"
    )


main()


Loading Stage-1 and Stage-2 models on mps...


/var/folders/gd/p6nf_1w11938t89t0bj2r9lc0000gn/T/ipykernel_66231/2472331833.py:231: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(mask_uint8, mode="L").save(output_path)


[1/2] 00000139_217557BB_1.png | normal DSC=0.903 | oracle DSC=0.932 | delta=+0.029


/var/folders/gd/p6nf_1w11938t89t0bj2r9lc0000gn/T/ipykernel_66231/2472331833.py:231: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(mask_uint8, mode="L").save(output_path)


[2/2] 00000155_A2330FD3_0.png | normal DSC=0.892 | oracle DSC=0.889 | delta=-0.003

Done. Comparison overlays: /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/oracle_roi_experiment/comparison_overlays
Metrics: /Users/JanayeCheong/Documents/radiomics_segmentation_models/TNSCUI2020-Seg-Rank1st/oracle_roi_experiment/oracle_roi_metrics.csv


In [4]:
from pathlib import Path

repo = Path("/content/sgh-segmodel")

matches = list(repo.rglob("segmentation_models_pytorch_4TorchLessThan120*"))

for match in matches:
    print(match)